In [ ]:
# ==============================
# Lung ViT Training + Full Metrics, Early Stopping, Test Inference (Research-Ready)
# ==============================
import os, random, time, math, json, csv
from collections import defaultdict, Counter

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision import datasets
from PIL import Image
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, roc_curve, auc
)
from itertools import cycle

# ------------------------------
# 0. Repro & Fast I/O
# ------------------------------
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(42)

# ------------------------------
# 1. Config
# ------------------------------
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 50            # max; early stopping will cut earlier
PATIENCE   = 5             # <-- early stopping patience
TARGET_PER_CLASS = 1000
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cost assumption (set your rate, e.g., 0.35 USD/hour for T4)
COST_PER_GPU_HOUR_TRAIN = 0.0
COST_PER_GPU_HOUR_TEST  = 0.0

# Where to save plots & outputs
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Toggle LIME (requires lime installed)
DO_LIME = False

# ------------------------------
# 2. Dataset Classes (with return_path for test inference)
# ------------------------------
class LungDataset(Dataset):
    def __init__(self, root_dir, transform=None, selected_indices=None, return_path=False):
        self.base = datasets.ImageFolder(root=root_dir)
        self.samples = self.base.samples
        self.classes = self.base.classes
        self.class_to_idx = self.base.class_to_idx
        self.transform = transform
        self.return_path = return_path
        if selected_indices:
            self.samples = [self.samples[i] for i in selected_indices]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform: image = self.transform(image)
        if self.return_path:
            return image, label, path
        return image, label

class AugmentedLungDataset(Dataset):
    def __init__(self, original_dataset, aug_dir, transform=None, selected_indices=None, return_path=False):
        self.samples = original_dataset.samples.copy()
        self.transform = transform
        self.class_to_idx = original_dataset.class_to_idx
        self.return_path = return_path

        for root, _, files in os.walk(aug_dir):
            cls_name = os.path.basename(root)
            if cls_name in self.class_to_idx:
                lbl = self.class_to_idx[cls_name]
                for file in files:
                    if file.lower().endswith((".jpg", ".jpeg", ".png")):
                        self.samples.append((os.path.join(root, file), lbl))

        if selected_indices:
            self.samples = [self.samples[i] for i in selected_indices]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform: image = self.transform(image)
        if self.return_path:
            return image, label, path
        return image, label

# ------------------------------
# 3. Transforms
# ------------------------------
train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.3, hue=0.02),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ------------------------------
# 4. Paths
# ------------------------------
DATA_ROOT = "DataSet"  # <-- Update if needed
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "valid")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

# ------------------------------
# 5. Base Datasets
# ------------------------------
train_base = LungDataset(TRAIN_DIR, transform=None)
val_base   = LungDataset(VAL_DIR, transform=None)
test_base  = LungDataset(TEST_DIR, transform=None)

NUM_CLASSES = len(train_base.classes)
cls_names   = train_base.classes
print(f"✅ Found {NUM_CLASSES} classes: {cls_names}")

# ------------------------------
# 6. Data Augmentation to balance (save to disk)
# ------------------------------
class_to_idxs = defaultdict(list)
for i, (path, lbl) in enumerate(train_base.samples):
    class_to_idxs[lbl].append((i, path))

AUGMENT_DIR = os.path.join(DATA_ROOT, "augmented")
os.makedirs(AUGMENT_DIR, exist_ok=True)
image_id = 0

augmentations = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3),
])

for lbl, idx_paths in class_to_idxs.items():
    cls_name = train_base.classes[lbl]
    save_dir = os.path.join(AUGMENT_DIR, cls_name)
    os.makedirs(save_dir, exist_ok=True)

    original_count = len(idx_paths)
    required = max(0, TARGET_PER_CLASS - original_count)
    sampled = random.choices(idx_paths, k=required) if required > 0 else []

    for _, path in sampled:
        img = Image.open(path).convert("RGB")
        img_aug = augmentations(img)
        save_path = os.path.join(save_dir, f"aug_{image_id}.jpg")
        img_aug.save(save_path)
        image_id += 1

# Final datasets & loaders
train_ds = AugmentedLungDataset(train_base, AUGMENT_DIR, train_transform, return_path=False)
val_ds   = LungDataset(VAL_DIR, eval_transform, return_path=False)
test_ds  = LungDataset(TEST_DIR, eval_transform, return_path=True)  # return_path for inference logging

pin = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True, pin_memory=pin)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,                   pin_memory=pin)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,                   pin_memory=pin)

print(f"📦 Final Train Dataset (with Augmentation): {len(train_ds)} samples")
print(f"📦 Dataset sizes - Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

# Quick balance (post-aug)
label_counts = Counter([lbl for _, lbl in train_ds.samples])
print("Class counts in train (post-aug):", {cls_names[k]: v for k, v in label_counts.items()})

# ------------------------------
# 7. ViT Model
# ------------------------------
def pair(t): return t if isinstance(t, tuple) else (t, t)

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim), nn.Dropout(dropout)
        )
    def forward(self, x): return self.net(x)

class Attention(nn.Module):
    def __init__(self, dim, heads=8, dim_head=64, dropout=0.):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.norm = nn.LayerNorm(dim)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.attend = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)
        self.to_out = nn.Sequential(nn.Linear(inner_dim, dim), nn.Dropout(dropout))

    def forward(self, x):
        x = self.norm(x)
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.heads), qkv)
        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        attn = self.attend(dots)
        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.layers = nn.ModuleList([
            nn.ModuleList([
                Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout),
                FeedForward(dim, mlp_dim, dropout=dropout)
            ]) for _ in range(depth)
        ])
    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x
        return self.norm(x)

class ViT(nn.Module):
    def __init__(self, *, image_size, patch_size, num_classes, dim, depth,
                 heads, mlp_dim, pool='cls', channels=3, dim_head=64,
                 dropout=0., emb_dropout=0.):
        super().__init__()
        image_height, image_width = pair(image_size)
        patch_height, patch_width = pair(patch_size)
        assert image_height % patch_height == 0 and image_width % patch_width == 0

        num_patches = (image_height // patch_height) * (image_width // patch_width)
        patch_dim = channels * patch_height * patch_width

        self.to_patch_embedding = nn.Sequential(
            Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
                      p1=patch_height, p2=patch_width),
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, dim),
            nn.LayerNorm(dim),
        )

        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.dropout = nn.Dropout(emb_dropout)
        self.transformer = Transformer(dim, depth, heads, dim_head, mlp_dim, dropout)

        self.pool = pool
        self.to_latent = nn.Identity()
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, img):
        x = self.to_patch_embedding(img)
        b, n, _ = x.shape
        cls_tokens = repeat(self.cls_token, '1 1 d -> b 1 d', b=b)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embedding[:, :(n + 1)]
        x = self.dropout(x)
        x = self.transformer(x)
        x = x.mean(dim=1) if self.pool == 'mean' else x[:, 0]
        x = self.to_latent(x)
        return self.mlp_head(x)

# Instantiate
model_name = "ViT"
model = ViT(
    image_size=IMAGE_SIZE, patch_size=16, num_classes=NUM_CLASSES,
    dim=512, depth=6, heads=8, mlp_dim=1024, dropout=0.1, emb_dropout=0.1
).to(DEVICE)

# ------------------------------
# 8. Optim, Sched, Checkpoint, Early Stopping
# ------------------------------
CHECKPOINT_PATH = f"{model_name.lower()}_best_checkpoint.pth"
RESUME          = True  # resume if checkpoint exists

criterion  = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(model.parameters(), lr=LR)
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

start_epoch   = 0
best_val_acc  = 0.0
epochs_no_improve = 0

if RESUME and os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    scheduler.load_state_dict(ckpt["sched_state"])
    best_val_acc = ckpt["best_val_acc"]
    start_epoch  = ckpt["epoch"] + 1
    print(f"✅ Resumed from epoch {ckpt['epoch']} | best_val_acc={best_val_acc:.4f}")
else:
    print("ℹ️  No checkpoint found or RESUME=False — starting fresh.")

# ------------------------------
# 9. Training Loop (+ Early Stopping)
# ------------------------------
history = {"epochs": [], "train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
epoch_times = []
train_start_time = time.time()

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\n🔄 Epoch {epoch+1}/{NUM_EPOCHS}")
    epoch_t0 = time.time()

    model.train()
    running_loss = correct_preds = total_preds = 0

    for images, labels in tqdm(train_loader, desc="  • Training"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct_preds += (outputs.argmax(1) == labels).sum().item()
        total_preds   += labels.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    train_acc  = correct_preds / total_preds
    print(f"    🟢 Train  | loss={train_loss:.4f}  acc={train_acc:.4f}")

    # Validation
    model.eval()
    val_loss = val_correct = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="  • Validate", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    val_loss /= len(val_loader.dataset)
    val_acc   = val_correct / len(val_loader.dataset)
    print(f"    🔵 Val    | loss={val_loss:.4f}  acc={val_acc:.4f}")

    history["epochs"].append(epoch + 1)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # Checkpoint & Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(
            {
                "epoch":        epoch,
                "model_state":  model.state_dict(),
                "optim_state":  optimizer.state_dict(),
                "sched_state":  scheduler.state_dict(),
                "best_val_acc": best_val_acc,
            },
            CHECKPOINT_PATH,
        )
        print(f"    💾 Saved new best checkpoint (acc={best_val_acc:.4f})")
    else:
        epochs_no_improve += 1
        print(f"    ⏳ No improvement for {epochs_no_improve}/{PATIENCE} epochs")
        if epochs_no_improve >= PATIENCE:
            print("🛑 Early stopping triggered.")
            break

    scheduler.step()
    print(f"    🔄 LR stepped -> {scheduler.get_last_lr()[0]:.6f}")

    epoch_times.append(time.time() - epoch_t0)

train_total_time = time.time() - train_start_time
print("\n🎉 Training complete.")
print(f"⏱️ Total training time: {train_total_time:.2f}s  (~{train_total_time/3600:.3f}h)")

# Reload best
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE)["model_state"])
model.eval()

# ------------------------------
# 10. Helpers
# ------------------------------
def get_preds_probs(loader, return_paths=False):
    all_logits, all_probs, all_preds, all_labels, all_paths = [], [], [], [], []
    with torch.no_grad():
        for batch in loader:
            if return_paths:
                images, labels, paths = batch
                all_paths.extend(paths)
            else:
                images, labels = batch
            images = images.to(DEVICE)
            logits = model(images)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = logits.argmax(1).cpu().numpy()
            all_logits.append(logits.cpu().numpy())
            all_probs.append(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    out = (
        np.concatenate(all_logits, axis=0),
        np.concatenate(all_probs, axis=0),
        np.array(all_preds),
        np.array(all_labels),
    )
    if return_paths:
        return out + (all_paths,)
    return out

def specificity_per_class(y_true, y_pred, num_classes):
    specs = []
    for c in range(num_classes):
        tp = np.sum((y_true == c) & (y_pred == c))
        tn = np.sum((y_true != c) & (y_pred != c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specs.append(spec)
    return np.array(specs), float(np.mean(specs))

def measure_inference_time(loader, n_batches=20):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            images = batch[0] if isinstance(batch, (list, tuple)) else batch
            if i >= n_batches: break
            images = images.to(DEVICE)
            t0 = time.time()
            _ = model(images)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            times.append(time.time() - t0)
    if not times: return 0.0
    avg_per_batch = float(np.mean(times))
    return avg_per_batch / images.size(0)

# ------------------------------
# 11. Collect metrics for Train / Val / Test
# ------------------------------
print("🔎 Collecting predictions for Train/Val/Test …")
_, train_probs, train_preds, train_labels = get_preds_probs(train_loader)
_, val_probs,   val_preds,   val_labels   = get_preds_probs(val_loader)

test_eval_t0 = time.time()
_, test_probs,  test_preds,  test_labels, test_paths = get_preds_probs(test_loader, return_paths=True)
test_total_time = time.time() - test_eval_t0

# Accuracy
train_acc_final = float((train_preds == train_labels).mean())
val_acc_final   = float((val_preds   == val_labels).mean())
test_acc_final  = float((test_preds  == test_labels).mean())

# Precision/Recall/F1 (macro in multiclass; binary ok)
avg_kw = {"average": "macro"} if NUM_CLASSES > 2 else {}
prec_test = float(precision_score(test_labels, test_preds, **avg_kw, zero_division=0))
rec_test  = float(recall_score(test_labels, test_preds, **avg_kw, zero_division=0))
f1_test   = float(f1_score(test_labels, test_preds, **avg_kw, zero_division=0))

# Specificity & Sensitivity (Recall) per class + macro
specs_cls, spec_macro = specificity_per_class(test_labels, test_preds, NUM_CLASSES)
sens_cls = recall_score(test_labels, test_preds, labels=list(range(NUM_CLASSES)),
                        average=None, zero_division=0)
sens_macro = float(sens_cls.mean())

# FPR per class
fpr_cls = []
for c in range(NUM_CLASSES):
    tn = np.sum((test_labels != c) & (test_preds != c))
    fp = np.sum((test_labels != c) & (test_preds == c))
    fpr_cls.append(float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0)

# Times & Costs
avg_infer_time = float(measure_inference_time(test_loader))
train_hours = train_total_time / 3600.0
test_hours  = test_total_time  / 3600.0
training_cost = float(train_hours * COST_PER_GPU_HOUR_TRAIN)
testing_cost  = float(test_hours  * COST_PER_GPU_HOUR_TEST)

print(f"✅ Final Accuracies | Train={train_acc_final:.4f}  Val={val_acc_final:.4f}  Test={test_acc_final:.4f}")
print(f"⏱️ Test evaluation time: {test_total_time:.2f}s  (~{test_hours:.3f}h)")
print(f"🧮 Avg inference time/sample (test): {avg_infer_time*1000:.2f} ms")
print(f"💰 Training cost: ${training_cost:.4f} | Testing cost: ${testing_cost:.4f}")

# ------------------------------
# 12. Save per-image Test Inference CSV
# ------------------------------
idx_to_class = {v: k for k, v in train_base.class_to_idx.items()}
preds_csv = os.path.join(FIG_DIR, f"{model_name}_test_inference_predictions.csv")
with open(preds_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    header = ["file", "true_label_idx", "true_label", "pred_label_idx", "pred_label", "top1_prob", "probs_json"]
    writer.writerow(header)
    for path, y_true, y_pred, probs in zip(test_paths, test_labels, test_preds, test_probs):
        writer.writerow([
            os.path.relpath(path, TEST_DIR),
            int(y_true), idx_to_class[int(y_true)],
            int(y_pred), idx_to_class[int(y_pred)],
            float(probs[int(y_pred)]),
            json.dumps([float(p) for p in probs]),
        ])
print(f"📝 Saved per-image test predictions → {preds_csv}")

# ------------------------------
# 13. PLOTS
# ------------------------------
# Helper: ROC with macro + per-class
def plot_roc_curves(probs, labels, title, save_path):
    n_classes = probs.shape[1]
    y_true_bin = np.eye(n_classes)[labels]  # one-hot
    fpr = dict(); tpr = dict(); roc_auc = dict()

    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    plt.figure()
    plt.plot(fpr["macro"], tpr["macro"], lw=2, label=f"{model_name} (macro AUC = {roc_auc['macro']:.3f})")
    colors = cycle(['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
                    '#9467bd', '#8c564b', '#e377c2', '#7f7f7f',
                    '#bcbd22', '#17becf'])
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], lw=1, label=f"{cls_names[i]} (AUC = {roc_auc[i]:.3f})")
    plt.plot([0,1],[0,1], lw=1, linestyle="--")
    plt.xlim([0.0,1.0]); plt.ylim([0.0,1.05])
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200); plt.close()
    return float(roc_auc["macro"])

# 1) Training Accuracy vs Testing Accuracy – per model
plt.figure()
plt.bar([f"{model_name}-Train", f"{model_name}-Test"], [train_acc_final, test_acc_final])
plt.ylabel("Accuracy"); plt.title("Training Accuracy vs Testing Accuracy (per model)")
plt.ylim(0,1); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"{model_name}_1_train_vs_test_acc_per_model.png"), dpi=200); plt.close()

# 2) Specificity vs Sensitivity – macro (Test)
plt.figure()
x = np.arange(1); width = 0.35
plt.bar(x - width/2, [spec_macro], width, label="Specificity")
plt.bar(x + width/2, [sens_macro], width, label="Sensitivity (Recall)")
plt.xticks(x, [model_name]); plt.ylim(0,1); plt.ylabel("Score")
plt.title("Specificity vs Sensitivity (Test, macro)")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"{model_name}_2_specificity_vs_sensitivity.png"), dpi=200); plt.close()

# 3) Precision, Recall, F1 – bars (Test macro)
plt.figure()
metrics = ["Precision", "Recall", "F1"]
vals = [prec_test, rec_test, f1_test]
plt.bar(metrics, vals); plt.ylim(0,1); plt.ylabel("Score")
plt.title("Precision / Recall / F1 (Test, macro)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"{model_name}_3_prf1_per_model.png"), dpi=200); plt.close()

# 4) Confusion Matrix — Test
cm_test = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=cls_names)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax, colorbar=False)
plt.title("Confusion Matrix — Test")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"{model_name}_4_confusion_matrix_test.png"), dpi=200)
plt.close()

# 4b) Confusion Matrix — Train (optional but useful)
cm_train = confusion_matrix(train_labels, train_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_train, display_labels=cls_names)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Greens', ax=ax, colorbar=False)
plt.title("Confusion Matrix — Train")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"{model_name}_4b_confusion_matrix_train.png"), dpi=200)
plt.close()

# 5) Training vs Testing Accuracy — Comparison
plt.figure()
labels_cmp = [model_name]
train_vals = [train_acc_final]; test_vals  = [test_acc_final]
x = np.arange(len(labels_cmp)); width = 0.35
plt.bar(x - width/2, train_vals, width, label="Train Acc")
plt.bar(x + width/2, test_vals,  width, label="Test Acc")
plt.xticks(x, labels_cmp); plt.ylim(0,1); plt.ylabel("Accuracy")
plt.title("Training vs Testing Accuracy — Model Comparison")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"{model_name}_5_train_vs_test_acc_comparison.png"), dpi=200); plt.close()

# 6) Training Time vs Avg Inference Time — grouped bar
plt.figure()
x = np.arange(1); width = 0.35
plt.bar(x - width/2, [train_total_time], width, label="Training Time (s)")
plt.bar(x + width/2, [avg_infer_time], width, label="Avg Inference Time / sample (s)")
plt.xticks(x, [model_name]); plt.ylabel("Seconds")
plt.title("Training Time vs Inference Time")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"{model_name}_6_time_train_vs_infer.png"), dpi=200); plt.close()

# 7) Training & Testing Cost — bars
plt.figure()
plt.bar([f"{model_name}-Train", f"{model_name}-Test"], [training_cost, testing_cost])
plt.ylabel("Cost (USD)")
plt.title("Training & Testing Cost")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"{model_name}_7_costs.png"), dpi=200); plt.close()

# 8) Training & Validation Accuracy vs Epochs — line
plt.figure()
plt.plot(history["epochs"], history["train_acc"], label="Train Acc")
plt.plot(history["epochs"], history["val_acc"],   label="Val Acc")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.ylim(0,1)
plt.title("Training & Validation Accuracy vs Epochs")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"{model_name}_8_train_val_acc_vs_epochs.png"), dpi=200); plt.close()

# 9) ROC — Training (macro + per-class)
auc_train_macro = plot_roc_curves(train_probs, train_labels,
                title=f"ROC Curve — Training ({model_name})",
                save_path=os.path.join(FIG_DIR, f"{model_name}_9_roc_training.png"))

# 10) ROC — Testing (macro + per-class)
auc_test_macro = plot_roc_curves(test_probs, test_labels,
                title=f"ROC Curve — Testing ({model_name})",
                save_path=os.path.join(FIG_DIR, f"{model_name}_10_roc_testing.png"))

# 11) TPR vs FPR — per class bars (Test)
plt.figure()
indices = np.arange(NUM_CLASSES); width = 0.35
plt.bar(indices - width/2, sens_cls, width, label="TPR (Recall)")
plt.bar(indices + width/2, fpr_cls,  width, label="FPR")
plt.xticks(indices, cls_names, rotation=20)
plt.ylim(0,1); plt.ylabel("Rate")
plt.title("TPR vs FPR per Class (Test)")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"{model_name}_11_tpr_vs_fpr_per_class.png"), dpi=200); plt.close()

print(f"📊 Saved all figures to: {FIG_DIR}/")

# ------------------------------
# 14. Classification Report (Test)
# ------------------------------
print("\n🧾 Classification Report (Test):\n",
      classification_report(test_labels, test_preds, target_names=cls_names, zero_division=0))

# ------------------------------
# 15. Save Summary Metrics CSV (for paper tables)
# ------------------------------
metrics_csv = os.path.join(FIG_DIR, f"{model_name}_summary_metrics.csv")
with open(metrics_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerow(["train_acc",        train_acc_final])
    writer.writerow(["val_acc",          val_acc_final])
    writer.writerow(["test_acc",         test_acc_final])
    writer.writerow(["precision_macro",  prec_test])
    writer.writerow(["recall_macro",     rec_test])
    writer.writerow(["f1_macro",         f1_test])
    writer.writerow(["specificity_macro", spec_macro])
    writer.writerow(["sensitivity_macro", sens_macro])
    writer.writerow(["auc_macro_train",  auc_train_macro])
    writer.writerow(["auc_macro_test",   auc_test_macro])
    writer.writerow(["train_total_time_s", train_total_time])
    writer.writerow(["test_total_time_s",  test_total_time])
    writer.writerow(["avg_infer_time_per_sample_s", avg_infer_time])
    writer.writerow(["training_cost_usd", training_cost])
    writer.writerow(["testing_cost_usd",  testing_cost])
print(f"🧮 Saved summary metrics → {metrics_csv}")

# ------------------------------
# 16. Optional: LIME (flip DO_LIME=True if you installed lime)
# ------------------------------
if DO_LIME:
    try:
        from lime import lime_image
        from skimage.segmentation import mark_boundaries

        raw_test_ds = LungDataset(TEST_DIR, transform=None)
        sample_img, sample_label = raw_test_ds[0]
        img_np = np.array(sample_img)

        def predict_proba(images_np):
            model.eval()
            with torch.no_grad():
                images_resized = torch.nn.functional.interpolate(
                    torch.tensor(images_np).permute(0, 3, 1, 2).float(),
                    size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False
                ) / 255.0
                images_resized = T.Normalize([0.485, 0.456, 0.406],
                                             [0.229, 0.224, 0.225])(images_resized)
                images_resized = images_resized.to(DEVICE)
                outputs = model(images_resized)
                return outputs.softmax(1).cpu().numpy()

        explainer = lime_image.LimeImageExplainer()
        explanation = explainer.explain_instance(
            image=img_np,
            classifier_fn=predict_proba,
            top_labels=1,
            hide_color=0,
            num_samples=1000
        )
        temp, mask = explanation.get_image_and_mask(
            label=explanation.top_labels[0],
            positive_only=False,
            hide_rest=False,
            num_features=10,
            min_weight=0.0
        )
        plt.figure(figsize=(6, 6))
        plt.imshow(mark_boundaries(temp / 255.0, mask))
        plt.title(f"LIME — {model_name}, class: {cls_names[sample_label]}")
        plt.axis('off'); plt.tight_layout()
        lime_path = os.path.join(FIG_DIR, f"{model_name}_lime_example.png")
        plt.savefig(lime_path, dpi=200); plt.close()
        print(f"🟢 LIME saved → {lime_path}")
    except Exception as e:
        print(f"⚠️ LIME skipped: {e}")
